In [2]:
import pandas as pd
import re

AUG_ONLY_FILE = "train_br_only_augmented.xlsx"
COMBINED_FILE = "train_br_augmented.xlsx"

OUTPUT_AUG_SAFE = "train_br_only_augmented_safe.xlsx"
OUTPUT_COMBINED_SAFE = "train_br_augmented_safe.xlsx"

TEXT_COL = "text_ar"
GLOSS_COL = "gloss_ar"

aug_df = pd.read_excel(AUG_ONLY_FILE)
combined_df = pd.read_excel(COMBINED_FILE)

print("Aug only shape   :", aug_df.shape)
print("Combined shape   :", combined_df.shape)

Aug only shape   : (28, 22)
Combined shape   : (130, 22)


In [3]:
def normalize_ar(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for col in ["replaced_token_original", "replaced_token_new", TEXT_COL, GLOSS_COL]:
    if col in aug_df.columns:
        aug_df[col] = aug_df[col].astype(str).apply(normalize_ar)

In [4]:
SAFE_BR_PAIRS = {
    ("الاكل", "الطعام"),
    ("الاكل", "وجبة"),
    ("الطعام", "الاكل"),
    ("الطعام", "وجبة"),
    ("وجبة", "الاكل"),
    ("وجبة", "الطعام"),
}

In [5]:
def is_safe_pair(row):
    old_tok = normalize_ar(row["replaced_token_original"])
    new_tok = normalize_ar(row["replaced_token_new"])
    return (old_tok, new_tok) in SAFE_BR_PAIRS

aug_safe_df = aug_df[aug_df.apply(is_safe_pair, axis=1)].copy()

print("Original augmented rows:", len(aug_df))
print("Safe augmented rows    :", len(aug_safe_df))

Original augmented rows: 28
Safe augmented rows    : 20


In [6]:
before_dedup = len(aug_safe_df)

aug_safe_df = aug_safe_df.drop_duplicates(subset=[TEXT_COL, GLOSS_COL]).reset_index(drop=True)

after_dedup = len(aug_safe_df)

print("Before dedup:", before_dedup)
print("After dedup :", after_dedup)

Before dedup: 20
After dedup : 20


In [7]:
preview_cols = [
    "parent_id",
    TEXT_COL,
    GLOSS_COL,
    "replaced_token_original",
    "replaced_token_new",
    "replacement_score",
    "replacement_source"
]

aug_safe_df[preview_cols].head(30)

,parent_id,text_ar,gloss_ar,replaced_token_original,replaced_token_new,replacement_score,replacement_source
0,2,حبة 3 مرة قبل الطعام بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,الاكل,الطعام,1.000000,manual
1,2,حبة 3 مرة قبل وجبة بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,الاكل,وجبة,1.000000,manual
2,3,حبة 3 مرة قبل الطعام .,3 مرة اكل قبل حبة,الاكل,الطعام,1.000000,manual
3,3,حبة 3 مرة قبل وجبة .,3 مرة اكل قبل حبة,الاكل,وجبة,1.000000,manual
4,4,حبة 3 مرة قبل الطعام بساعة .,3 مرة اكل قبل ساعة حبة,الاكل,الطعام,0.011844,fill_mask
5,4,حبة 3 مرة قبل وجبة بساعة .,3 مرة اكل قبل ساعة حبة,الاكل,وجبة,1.000000,manual
6,9,خده ورا الطعام,اكل بعد حبة تمام معدة فاضية لا,الاكل,الطعام,1.000000,manual
7,9,خده ورا وجبة,اكل بعد حبة تمام معدة فاضية لا,الاكل,وجبة,1.000000,manual
8,10,بعد الطعام,اكل خلص فورا دواء تمام,الاكل,الطعام,1.000000,manual
9,10,بعد وجبة,اكل خلص فورا دواء تمام,الاكل,وجبة,1.000000,manual


In [8]:
if "source_type" in combined_df.columns:
    base_df = combined_df[combined_df["source_type"] == "original"].copy()
else:
    raise ValueError("Column 'source_type' not found in combined file.")

print("Base rows:", len(base_df))

Base rows: 102


In [9]:
combined_safe_df = pd.concat([base_df, aug_safe_df], ignore_index=True)

print("Base rows         :", len(base_df))
print("Safe augmented    :", len(aug_safe_df))
print("Combined safe rows:", len(combined_safe_df))

Base rows         : 102
Safe augmented    : 20
Combined safe rows: 122


In [10]:
preferred_cols = [
    "sample_id",
    "id",
    "parent_id",
    "source_type",
    "aug_method",
    "is_augmented",
    TEXT_COL,
    GLOSS_COL,
    "text_norm",
    "gloss_norm",
    "tokens_text",
    "tokens_gloss",
    "masked_text",
    "replace_position",
    "replaced_token_original",
    "replaced_token_new",
    "replacement_score",
    "replacement_source",
    "replaceable_positions",
    "replaceable_tokens",
    "protected_tokens_found",
    "br_ready"
]

existing_cols = [c for c in preferred_cols if c in combined_safe_df.columns]
remaining_cols = [c for c in combined_safe_df.columns if c not in existing_cols]

combined_safe_df = combined_safe_df[existing_cols + remaining_cols]
aug_safe_df = aug_safe_df[[c for c in existing_cols + remaining_cols if c in aug_safe_df.columns]]

combined_safe_df.head(20)

,sample_id,id,parent_id,source_type,aug_method,is_augmented,text_ar,gloss_ar,text_norm,gloss_norm,...,masked_text,replace_position,replaced_token_original,replaced_token_new,replacement_score,replacement_source,replaceable_positions,replaceable_tokens,protected_tokens_found,br_ready
0,1,1,1,original,none,0,انتبه,انتبه خطر تمام,انتبه,انتبه خطر تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],['انتبه'],0
1,2,2,2,original,none,0,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,...,NaN,NaN,NaN,NaN,NaN,NaN,[4],['الاكل'],"['بنص', 'حبة', 'ساعة', 'قبل', 'مرة']",1
2,3,3,3,original,none,0,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,...,NaN,NaN,NaN,NaN,NaN,NaN,[4],['الاكل'],"['حبة', 'قبل', 'مرة']",1
3,4,4,4,original,none,0,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,...,NaN,NaN,NaN,NaN,NaN,NaN,[4],['الاكل'],"['بساعة', 'حبة', 'قبل', 'مرة']",1
4,6,6,6,original,none,0,6 شهر,6 شهر,6 شهر,6 شهر,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],['شهر'],0
5,7,7,7,original,none,0,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],"['بكل', 'نقطتين']",0
6,8,8,8,original,none,0,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],"['ثلاثة', 'لمدة', 'يوم']",0
7,9,9,9,original,none,0,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,...,NaN,NaN,NaN,NaN,NaN,NaN,[2],['الاكل'],['ورا'],1
8,10,10,10,original,none,0,بعد الاكل,اكل خلص فورا دواء تمام,بعد الاكل,اكل خلص فورا دواء تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[1],['الاكل'],['بعد'],1
9,13,13,13,original,none,0,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,...,NaN,NaN,NaN,NaN,NaN,NaN,[],[],"['اسهال', 'تقلق', 'شوية', 'لا', 'ممكن', 'يعملك']",0


In [11]:
aug_safe_df.to_excel(OUTPUT_AUG_SAFE, index=False)
combined_safe_df.to_excel(OUTPUT_COMBINED_SAFE, index=False)

print("Saved:")
print("-", OUTPUT_AUG_SAFE)
print("-", OUTPUT_COMBINED_SAFE)

Saved:
- train_br_only_augmented_safe.xlsx
- train_br_augmented_safe.xlsx


In [12]:
print("Unique safe pairs:")
print(
    aug_safe_df[["replaced_token_original", "replaced_token_new"]]
    .drop_duplicates()
    .sort_values(by=["replaced_token_original", "replaced_token_new"])
    .to_string(index=False)
)

Unique safe pairs:
replaced_token_original replaced_token_new
                  الاكل             الطعام
                  الاكل               وجبة
                   وجبة              الاكل
                   وجبة             الطعام
